# Evaluation Dashboard: 3-Fold Walk-Forward Validation
This notebook visualizes the performance of the forecasting pipeline across three distinct historical regimes: 2020, 2021, and 2022.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error

# Add project root to sys.path
PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from src.config import Config
from src.training.pipeline import ForecastingPipeline

In [2]:
def run_evaluation():
    raw_sales = pd.read_parquet(PROJECT_ROOT / Config.SALES_TRAIN_FILE)
    raw_sales['Date'] = pd.to_datetime(raw_sales['Date'])
    
    folds = [
        {'train_max_year': 2019, 'test_start': 2020, 'test_end': 2020, 'weight': 0.2},
        {'train_max_year': 2020, 'test_start': 2021, 'test_end': 2021, 'weight': 0.3},
        {'train_max_year': 2021, 'test_start': 2022, 'test_end': 2022, 'weight': 0.5},
    ]
    
    all_results = []
    
    for i, fold in enumerate(folds):
        fold_num = i + 1
        train_end_date = pd.to_datetime(f"{fold['train_max_year']}-12-31")
        test_start_date = pd.to_datetime(f"{fold['test_start']}-01-01")
        test_end_date = pd.to_datetime(f"{(fold['test_end'])}-12-31")
        
        train_df = raw_sales[raw_sales['Date'] <= train_end_date].copy()
        test_df = raw_sales[(raw_sales['Date'] >= test_start_date) & (raw_sales['Date'] <= test_end_date)].copy()
        
        # Naive Baseline (LY)
        test_df['ly_Revenue'] = test_df['Date'].map(raw_sales.set_index(raw_sales['Date'] + pd.Timedelta(days=364))['Revenue']).ffill()
        
        pipeline = ForecastingPipeline()
        pipeline.fit(train_df)
        
        predictions = pipeline.predict(test_df[['Date']])
        test_df['p_Revenue'] = predictions['Revenue'].values
        test_df['p_COGS'] = predictions['COGS'].values
        test_df['p_Profit'] = test_df['p_Revenue'] - test_df['p_COGS']
        test_df['Profit'] = test_df['Revenue'] - test_df['COGS']
        
        all_results.append({
            'fold': fold_num,
            'period': fold['test_start'],
            'df': test_df,
            'weight': fold['weight']
        })
        
    return all_results

results = run_evaluation()

Discovering Category-Specific Signals & COGS Profiles...
Calculating Data-driven Sample Weights...
Training Normalized Revenue Model...
Starting Optimized Signal Discovery...
Discovery complete: Found 70 pure event signals.
Starting Category Profile Discovery...
Category discovery complete: Profiles for 12 months created.
Peak Momentum Discovered: 2.50x (from 2019-06-29)
Training COGS Ratio Model...
Starting Dynamic Inertia Weight Discovery...
Learned Inertia Weights: Rev=0.121, Order=0.830, AOV=-0.709
Inertia Confidence (R2): 1.000 -> Trust Weight: 0.95
Data-driven Damping: 0.920 (based on YoY volatility 0.161)
Dynamic Inertia Applied (Calibrated M: 0.855x)
Generating stationary recursive forecast (Optimized)...
Discovering Category-Specific Signals & COGS Profiles...
Calculating Data-driven Sample Weights...
Training Normalized Revenue Model...
Starting Optimized Signal Discovery...
Discovery complete: Found 71 pure event signals.
Starting Category Profile Discovery...
Category disco

## Visual Comparison: Actual vs Predicted vs Naive

In [3]:
for res in results:
    df = res['df']
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=df['Date'], y=df['Revenue'], name='Actual Revenue', line=dict(color='royalblue', width=2)))
    fig.add_trace(go.Scatter(x=df['Date'], y=df['p_Revenue'], name='Predicted Revenue', line=dict(color='firebrick', width=2, dash='dot')))
    fig.add_trace(go.Scatter(x=df['Date'], y=df['ly_Revenue'], name='Naive (Last Year)', line=dict(color='gray', width=1, dash='dash'), opacity=0.5))
    
    fig.update_layout(
        title=f"Fold {res['fold']} Performance ({res['period']}) - Revenue",
        xaxis_title="Date",
        yaxis_title="Revenue",
        hovermode="x unified",
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.show()

## Gross Profit Accuracy

In [4]:
for res in results:
    df = res['df']
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=df['Date'], y=df['Profit'], name='Actual Profit', line=dict(color='seagreen', width=2)))
    fig.add_trace(go.Scatter(x=df['Date'], y=df['p_Profit'], name='Predicted Profit', line=dict(color='orange', width=2, dash='dot')))
    
    fig.update_layout(
        title=f"Fold {res['fold']} Performance ({res['period']}) - Gross Profit",
        xaxis_title="Date",
        yaxis_title="Profit",
        hovermode="x unified",
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.show()

## Final Summary Metrics

In [5]:
summary_data = []
for res in results:
    df = res['df']
    mae_rev = mean_absolute_error(df['Revenue'], df['p_Revenue'])
    mae_profit = mean_absolute_error(df['Profit'], df['p_Profit'])
    mae_ly = mean_absolute_error(df['Revenue'], df['ly_Revenue'].fillna(0))
    lift = (1 - mae_rev / mae_ly) * 100 if mae_ly > 0 else 0
    
    summary_data.append({
        'Fold': res['fold'],
        'Period': res['period'],
        'MAE Revenue': f"{mae_rev:,.0f}",
        'MAE Profit': f"{mae_profit:,.0f}",
        'Lift vs LY (%)': f"{lift:.2f}%",
        'Weight': res['weight']
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

,Fold,Period,MAE Revenue,MAE Profit,Lift vs LY (%),Weight
0,1,2020,"629,712","127,290",32.19%,0.2
1,2,2021,"657,152","127,343",15.33%,0.3
2,3,2022,"599,322","118,815",30.29%,0.5
